# Secrets Administration — NeoBank

**One-time, manual, administrator-only notebook. No job references it. Never run as part of the pipeline.**

This notebook creates the secret scope `banking-scope` and stores the three pipeline secrets. All values are entered interactively via `getpass` prompts — they are **never written in code, never printed, and never committed**.

Secrets stored:

| Key | Purpose |
| --- | --- |
| `neon-username` | Neon PostgreSQL login user |
| `neon-password` | Neon PostgreSQL password |
| `gmail_app_password` | Gmail app password for the pipeline notification email |

Alternative (no code): the Databricks CLI can do the same — `databricks secrets create-scope banking-scope` and `databricks secrets put-secret banking-scope neon-username` etc.


In [0]:
# =====================================================
# CELL 1: CREATE SECRET SCOPE (one-time admin operation)
# =====================================================

import requests

secret_scope_name = "banking-scope"

ctx = (
    dbutils.notebook.entry_point
    .getDbutils()
    .notebook()
    .getContext()
)

api_url = ctx.apiUrl().getOrElse(None)
api_token = ctx.apiToken().getOrElse(None)

if not api_url or not api_token:
    raise Exception(
        "Workspace API credentials are unavailable. "
        "Use the Databricks CLI instead: databricks secrets create-scope banking-scope"
    )

headers = {
    "Authorization": f"Bearer {api_token}",
    "Content-Type": "application/json"
}

response = requests.post(
    f"{api_url}/api/2.0/secrets/scopes/create",
    headers=headers,
    json={"scope": secret_scope_name},
    timeout=30
)

if response.status_code == 200:
    print("Secret scope created successfully.")
elif "already exists" in response.text.lower():
    print("Secret scope already exists. Continuing.")
else:
    raise Exception(
        f"Scope creation failed: {response.status_code} - {response.text}"
    )

# Never print api_token.


In [0]:
# =====================================================
# CELL 2: STORE SECRETS (values entered interactively — never in code)
# =====================================================

import getpass

def put_secret(key, value):
    response = requests.post(
        f"{api_url}/api/2.0/secrets/put",
        headers=headers,
        json={
            "scope": secret_scope_name,
            "key": key,
            "string_value": value
        },
        timeout=30
    )
    if response.status_code != 200:
        raise Exception(
            f"Failed to store secret '{key}': "
            f"{response.status_code} - {response.text}"
        )
    print(f"Secret '{key}' stored successfully.")

put_secret("neon-username", getpass.getpass("Enter Neon database username: "))
put_secret("neon-password", getpass.getpass("Enter Neon database password: "))
put_secret(
    "gmail_app_password",
    getpass.getpass("Enter Gmail App Password: ").replace(" ", "")
)

In [0]:
# =====================================================
# CELL 3: VERIFY — list KEYS only. Never print values.
# =====================================================

keys = [s.key for s in dbutils.secrets.list(secret_scope_name)]
print("Secret keys present:", sorted(keys))

expected = {"neon-username", "neon-password", "gmail_app_password"}
missing = expected - set(keys)
if missing:
    raise Exception(f"Missing secrets: {missing}")

# Non-emptiness check WITHOUT printing any value
for key in sorted(expected):
    assert dbutils.secrets.get(scope=secret_scope_name, key=key) is not None
print("All secrets present and non-empty. Admin setup complete.")
